# 🧹 Notebook 02 — Data Preprocessing
**Step 5** | `notebooks/02_preprocessing.ipynb`

Goals:
- Handle missing values
- Remove duplicate records
- Fix and standardise data types
- Drop irrelevant/redundant columns
- Save cleaned data to `data/processed/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')
%matplotlib inline

os.makedirs('../data/processed', exist_ok=True)

## 1. Load Raw Data

In [ ]:
data = {
    'Order_ID':              [0, 1, 2, 3, 4, 5, 6],
    'Distance_km':           [5, 22, 7.93, 16.42, 19.52, 17.44, 19.03],
    'Weather':               ['Clear','Rainy','Windy','Clear','Foggy','Rainy','Clear'],
    'Traffic_Level':         ['Low','Medium','Low','Medium','Low','Medium','Low'],
    'Time_of_Day':           ['Morning','Evening','Afternoon','Evening','Night','Afternoon','Morning'],
    'Vehicle_Type':          ['Bike','Scooter','Scooter','Bike','Scooter','Scooter','Bike'],
    'Preparation_Time_min':  [10, 25, 12, 20, 28, 5, 16],
    'Courier_Experience_yrs':[3.0, 1.5, 1.0, 2.0, 1.0, 1.0, 5.0],
    'Delivery_Time_min':     [25, 55, 43, 84, 59, 36, 68],
}
df = pd.DataFrame(data)
print('Raw shape:', df.shape)
df.head()

## 2. Handle Missing Values
> Fill numeric with median, categorical with mode.

In [ ]:
# Check before
print('Missing before:')
print(df.isnull().sum())

# Simulate a missing value to test (comment out in real use)
df_test = df.copy()
df_test.loc[0, 'Distance_km'] = np.nan
df_test.loc[2, 'Weather'] = np.nan

# Fill numeric columns with median
num_cols = df_test.select_dtypes(include=np.number).columns
for col in num_cols:
    if df_test[col].isnull().any():
        median_val = df_test[col].median()
        df_test[col].fillna(median_val, inplace=True)
        print(f'  Filled {col} with median = {median_val}')

# Fill categorical columns with mode
cat_cols = df_test.select_dtypes(include='object').columns
for col in cat_cols:
    if df_test[col].isnull().any():
        mode_val = df_test[col].mode()[0]
        df_test[col].fillna(mode_val, inplace=True)
        print(f'  Filled {col} with mode = {mode_val}')

print('\nMissing after:', df_test.isnull().sum().sum())
df = df_test  # use filled version

## 3. Remove Duplicate Records

In [ ]:
print('Duplicates before:', df.duplicated().sum())
df = df.drop_duplicates()
df = df.reset_index(drop=True)
print('Duplicates after: ', df.duplicated().sum())
print('Shape after dedup:', df.shape)

## 4. Fix and Standardise Data Types

In [ ]:
print('dtypes before:')
print(df.dtypes)

df['Preparation_Time_min']   = df['Preparation_Time_min'].astype(float)
df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].astype(float)
df['Distance_km']            = df['Distance_km'].astype(float)
df['Delivery_Time_min']      = df['Delivery_Time_min'].astype(float)
df['Weather']                = df['Weather'].astype('category')
df['Traffic_Level']          = df['Traffic_Level'].astype('category')
df['Time_of_Day']            = df['Time_of_Day'].astype('category')
df['Vehicle_Type']           = df['Vehicle_Type'].astype('category')

print('\ndtypes after:')
print(df.dtypes)

## 5. Drop Irrelevant / Redundant Columns

In [ ]:
print('Columns before:', df.columns.tolist())

# Order_ID is an identifier — not a feature
df.drop(columns=['Order_ID'], inplace=True)

print('Columns after:', df.columns.tolist())

## 6. Outlier Check (IQR method)

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns
fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))

for ax, col in zip(axes, num_cols):
    ax.boxplot(df[col], patch_artist=True,
               boxprops=dict(facecolor='#4C9BE8', color='navy'))
    ax.set_title(col, fontsize=9)
    ax.tick_params(axis='x', labelbottom=False)

plt.suptitle('Outlier Check — Numeric Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Print IQR bounds
for col in num_cols:
    Q1, Q3 = df[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    print(f'{col}: [{lower:.2f}, {upper:.2f}]  outliers = {len(outliers)}')

## 7. Save Cleaned Data

In [ ]:
df.to_csv('../data/processed/delivery_processed.csv', index=False)
print('✅ Saved to data/processed/delivery_processed.csv')
print('Final shape:', df.shape)
df.head()